# Part 4 — Exercise solutions

⚠️ Run `part3_search.ipynb` and `part4_extract.ipynb` first — these reuse their
caches and definitions.

In [ ]:
import os
import time
from enum import Enum
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv("../../.env", override=True)
client = genai.Client()
MODEL = "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"
pd.set_option("display.max_colwidth", 100)

games = pd.read_csv("../../data/games.csv")
df = pd.read_csv("../../data/reviews.csv").merge(games, on="game_id")
doc_vecs = np.load("../cache/review_embeddings.npy")


class Issue(str, Enum):
    crashes_or_bugs = "crashes_or_bugs"
    performance = "performance"
    difficulty = "difficulty"
    monetization = "monetization"
    content_amount = "content_amount"
    multiplayer_or_netcode = "multiplayer_or_netcode"
    controls_or_ui = "controls_or_ui"
    other = "other"


def call_structured(text, schema, prompt="Analyze this game review:"):
    for attempt in range(7):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=f"{prompt}\n\n{text}",
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0.0,
                ),
            )
            return response.parsed
        except genai.errors.APIError:
            time.sleep(2**attempt)
    raise RuntimeError("failed after 7 attempts")

## 7.1 — Severity

In [ ]:
class ReviewAnalysisV2(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]
    issues: list[Issue]
    severity: Literal["minor", "major", "game_breaking"]
    summary: str


ten = df[df["game_id"] == "g12"].head(10)
for _, r in ten.iterrows():
    a = call_structured(r["review_text"], ReviewAnalysisV2)
    print(f"{a.severity:>14} · {a.summary}")

## 7.2 — Feature requests

In [ ]:
class WishlistAnalysis(BaseModel):
    feature_requests: list[str]


wishes = []
sample = df[df["game_id"].isin(["g01", "g05", "g09", "g16"])].groupby("game_id").head(10)
for _, r in sample.iterrows():
    a = call_structured(
        r["review_text"], WishlistAnalysis,
        prompt="List concrete features this reviewer ASKS FOR (not complaints). Empty list if none.",
    )
    wishes.append({"title": r["title"], "n_requests": len(a.feature_requests), "requests": a.feature_requests})

wish_df = pd.DataFrame(wishes)
print(wish_df.groupby("title")["n_requests"].sum().sort_values(ascending=False))
print("\nSample requests:", [w for reqs in wish_df["requests"].head(8) for w in reqs][:6])

## 7.3 — Cross-check the devs (Silent Depths)

In [ ]:
class ReviewAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]
    issues: list[Issue]
    summary: str


g12 = df[df["game_id"] == "g12"]
issue_list = []
for _, r in g12.iterrows():
    a = call_structured(r["review_text"], ReviewAnalysis)
    issue_list.extend(i.value for i in a.issues)

print(pd.Series(issue_list).value_counts())

In [ ]:
print(Path("../../data/docs/g12-patch-notes.md").read_text(encoding="utf-8"))

Compare: the dominant complaint cluster (crashes/bugs — the save corruption) is
addressed in the patch notes; check the *dates* — complaints should thin out
after the fix ships. You just correlated two data sources by hand; on Day 4 an
orchestrator agent does this join (reviews-agent + docs-agent → report).

## 7.4 — Mini-pipeline: search → analyze → verdict

In [ ]:
def embed_query(query):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768),
    )
    q = np.array(result.embeddings[0].values)
    return q / np.linalg.norm(q)


def pipeline(topic, top_k=8):
    scores = doc_vecs @ embed_query(topic)                 # 1. retrieve
    hits = df.iloc[np.argsort(scores)[::-1][:top_k]]

    analyzed = []
    for _, r in hits.iterrows():                            # 2. extract
        a = call_structured(r["review_text"], ReviewAnalysis)
        analyzed.append({"title": r["title"], "sentiment": a.sentiment, "summary": a.summary})

    report = pd.DataFrame(analyzed)
    print(f"📋 Topic: {topic!r}\n")                         # 3. report
    for title, group in report.groupby("title"):
        neg = (group["sentiment"] != "positive").mean()
        print(f"• {title}: {len(group)} relevant reviews, {neg:.0%} unhappy")
        for s in group["summary"]:
            print(f"    - {s}")


pipeline("crashes and technical problems")

**This is Day 5's demo, as a hardcoded script.** The difference between this and
an agent: here *you* decided the steps and their order. Starting next session, the
model reads a goal, sees its tools, and decides the steps itself.